# Etapa 1 — Python Sólido para el Proyecto RAG
### Notebook de práctica

---

**Cómo usar este notebook:**
- Ejecutá cada celda en orden con `Shift + Enter`.
- Leé la explicación antes de ver el código.
- Las celdas marcadas con **→ Tu turno** son ejercicios. Completalas vos.
- No avances si una celda falla. Entendé el error antes de continuar.

---

## Sección 1 — Programación Orientada a Objetos

En el proyecto, casi todo lo que toca LangChain son clases: los loaders, los splitters, los retrievers, el estado del grafo.  
Esta sección te da el modelo mental para leer y modificar ese código sin adivinar.

### 1.1 Tu primera clase — estructura básica

Una clase agrupa datos (atributos) y comportamiento (métodos) en una sola unidad.

- `__init__` se ejecuta automáticamente cuando se crea un objeto.
- `self` es la referencia al objeto actual. Sin `self`, el atributo no pertenece al objeto.
- Los métodos son funciones dentro de la clase que siempre reciben `self` como primer parámetro.

In [ ]:
# Ejecutá esta celda y observá la salida

class DocumentoRAG:
    """Representa un documento cargado en el sistema RAG."""

    def __init__(self, nombre: str, contenido: str):
        self.nombre = nombre
        self.contenido = contenido
        self.procesado = False  # estado interno del objeto

    def procesar(self) -> None:
        """Marca el documento como procesado."""
        self.procesado = True
        print(f"Documento '{self.nombre}' marcado como procesado.")

    def resumen(self) -> str:
        """Devuelve un resumen corto del documento."""
        estado = 'procesado' if self.procesado else 'pendiente'
        return f"[{estado}] {self.nombre} — {len(self.contenido)} caracteres"


# Crear dos instancias distintas del mismo molde
doc1 = DocumentoRAG("manual_tecnico.pdf", "Contenido del manual técnico...")
doc2 = DocumentoRAG("faq_clientes.txt", "Preguntas frecuentes...")

print(doc1.resumen())
print(doc2.resumen())

doc1.procesar()

print(doc1.resumen())  # ¿cambió el estado?
print(doc2.resumen())  # ¿se afectó doc2?

### → Tu turno 1.1

Creá una clase `BaseDeConocimiento` que:
- Reciba `nombre` (str) y `ruta` (str) en `__init__`
- Tenga un atributo `cargada` que empiece en `False`
- Tenga un método `cargar()` que cambie `cargada` a `True` e imprima un mensaje
- Tenga un método `estado()` que devuelva un string indicando si está cargada o no

Después creá dos instancias, cargá una y verificá que la otra no se vea afectada.

In [ ]:
# → Tu turno: escribí tu clase acá

class BaseDeConocimiento:
    pass  # reemplazá este pass con tu implementación


# Probá tu clase:
# base_legal = BaseDeConocimiento('legal', 'datos/legal/')
# base_tecnica = BaseDeConocimiento('tecnica', 'datos/tecnica/')
# base_legal.cargar()
# print(base_legal.estado())
# print(base_tecnica.estado())  # debe seguir como 'no cargada'

### 1.2 Herencia — cómo LangChain la usa

LangChain define clases base que vos extendés. Por ejemplo, `BaseRetriever` tiene métodos que todos los retrievers deben implementar.  
Cuando creás una subclase, heredás todo lo que ya tiene la clase padre y podés agregar o modificar comportamiento.

En el ejemplo siguiente simulamos este patrón sin depender de LangChain.

In [ ]:
# Clase base: define la interfaz común
class RetrieverBase:
    """Clase base para todos los retrievers del proyecto."""

    def __init__(self, nombre: str):
        self.nombre = nombre

    def buscar(self, query: str) -> list[str]:
        """Todas las subclases DEBEN implementar este método."""
        raise NotImplementedError("Las subclases deben implementar buscar()")

    def describir(self) -> str:
        return f"Retriever: {self.nombre} ({self.__class__.__name__})"


# Subclase concreta: implementa buscar() para un caso específico
class RetrieverSimulado(RetrieverBase):
    """Retriever que devuelve resultados simulados (útil para tests)."""

    def __init__(self, nombre: str, documentos: list[str]):
        super().__init__(nombre)  # llama al __init__ del padre
        self.documentos = documentos

    def buscar(self, query: str) -> list[str]:
        # Simulación: devuelve docs que contienen alguna palabra del query
        palabras = query.lower().split()
        return [
            doc for doc in self.documentos
            if any(p in doc.lower() for p in palabras)
        ]


# Probar la herencia
docs = [
    "El modelo de lenguaje procesa tokens de forma secuencial.",
    "Los embeddings representan texto en espacios vectoriales.",
    "RAG combina recuperación de información con generación de texto.",
    "FastAPI permite crear APIs asíncronas en Python.",
]

retriever = RetrieverSimulado("base_principal", docs)
print(retriever.describir())  # método heredado del padre

resultados = retriever.buscar("embeddings y vectores")
print(f"\nResultados para 'embeddings y vectores':")
for r in resultados:
    print(f"  - {r}")

### 1.3 TypedDict — el estado del agente en el proyecto

Esto aparece textualmente en el proyecto. `TypedDict` define un diccionario con claves y tipos fijos.  
LangGraph lo usa para el estado del grafo: cada nodo recibe este estado y devuelve una versión actualizada.

La anotación `Annotated[list, operator.add]` le dice a LangGraph que cuando un nodo agrega mensajes,  
debe **concatenarlos** a la lista existente en lugar de reemplazarla.

In [ ]:
from typing import TypedDict, Annotated
import operator

# Esta es la definición real del estado en el proyecto
class RAGState(TypedDict):
    messages: Annotated[list, operator.add]  # los mensajes se acumulan
    context: str                              # el contexto recuperado se reemplaza


# Crear un estado inicial
estado_inicial: RAGState = {
    "messages": [{"role": "user", "content": "¿Qué es RAG?"}],
    "context": ""
}

print("Estado inicial:")
print(f"  messages: {estado_inicial['messages']}")
print(f"  context: '{estado_inicial['context']}'")

# Simular lo que haría un nodo de recuperación
def nodo_recuperacion(state: RAGState) -> dict:
    contexto_recuperado = "RAG es una técnica que combina recuperación y generación."
    return {"context": contexto_recuperado}  # solo actualiza context

# Simular lo que haría un nodo de generación
def nodo_generacion(state: RAGState) -> dict:
    respuesta = f"Basado en el contexto: {state['context'][:50]}..."
    return {"messages": [{"role": "assistant", "content": respuesta}]}  # agrega al historial

# Ejecutar el flujo manualmente
estado = estado_inicial.copy()
estado.update(nodo_recuperacion(estado))
estado["messages"] = estado["messages"] + nodo_generacion(estado)["messages"]

print("\nEstado final:")
print(f"  context: '{estado['context']}'")
print(f"  messages ({len(estado['messages'])} total): {estado['messages']}")

---
## Sección 2 — Type Hints

En el proyecto, todas las funciones usan type hints. Son la documentación más confiable:  
no se desactualiza porque es parte del código.

In [ ]:
from typing import Optional, Union

# Tipos simples
def saludar(nombre: str) -> str:
    return f"Hola, {nombre}"

# Tipos compuestos — aparecen en el proyecto constantemente
def procesar_mensajes(mensajes: list[dict]) -> list[str]:
    """Extrae el contenido de cada mensaje."""
    return [m["content"] for m in mensajes]

# Optional: el parámetro puede ser None
def conectar_modelo(api_base: str, timeout: Optional[int] = None) -> bool:
    """Simula la conexión a un modelo. timeout=None usa el default del sistema."""
    print(f"Conectando a {api_base} (timeout: {timeout or 'default'})")
    return True

# Probar
mensajes_ejemplo = [
    {"role": "user", "content": "¿Qué es un embedding?"},
    {"role": "assistant", "content": "Un embedding es una representación vectorial del texto."},
]

contenidos = procesar_mensajes(mensajes_ejemplo)
for c in contenidos:
    print(c)

conectar_modelo("http://localhost:9087/v1")
conectar_modelo("http://localhost:9087/v1", timeout=30)

### → Tu turno 2.1

Escribí una función `filtrar_fragmentos` que:
- Reciba `fragmentos` (list[str]) y `longitud_minima` (int, por defecto 50)
- Devuelva una `list[str]` con solo los fragmentos que tengan al menos `longitud_minima` caracteres
- Use type hints en todos los parámetros y en el retorno

In [ ]:
# → Tu turno: completá la función

def filtrar_fragmentos(fragmentos: list[str], longitud_minima: int = 50) -> list[str]:
    pass  # reemplazá este pass


# Probá tu función con estos datos:
textos = [
    "Texto corto.",
    "Este fragmento es lo suficientemente largo como para ser útil en RAG.",
    "Otro texto breve.",
    "Los embeddings permiten representar el significado semántico de un texto en un espacio vectorial continuo.",
]

# resultado = filtrar_fragmentos(textos)
# print(f"De {len(textos)} fragmentos, {len(resultado)} superan los 50 caracteres.")
# for f in resultado:
#     print(f"  [{len(f)} chars] {f[:60]}..." if len(f) > 60 else f"  [{len(f)} chars] {f}")

---
## Sección 3 — Programación Asíncrona

FastAPI y el streaming de respuestas del modelo dependen de async/await.  
Esta sección te da el modelo mental necesario para trabajar en la capa de API.

In [ ]:
import asyncio
import time

# Primero: la versión síncrona — bloquea mientras espera
def llamar_modelo_sync(query: str) -> str:
    print(f"  [sync] Procesando: '{query}'")
    time.sleep(1)  # simula la latencia del modelo
    return f"Respuesta a: {query}"

print("=== Versión síncrona ===")
inicio = time.time()
r1 = llamar_modelo_sync("¿Qué es RAG?")
r2 = llamar_modelo_sync("¿Qué es un embedding?")
print(f"Tiempo total: {time.time() - inicio:.1f}s (procesó una por vez)")

In [ ]:
import asyncio
import time

# La versión asíncrona — libera el event loop mientras espera
async def llamar_modelo_async(query: str) -> str:
    print(f"  [async] Iniciando: '{query}'")
    await asyncio.sleep(1)  # simula latencia SIN bloquear
    print(f"  [async] Completado: '{query}'")
    return f"Respuesta a: {query}"

async def procesar_dos_queries():
    # gather ejecuta ambas corutinas concurrentemente
    r1, r2 = await asyncio.gather(
        llamar_modelo_async("¿Qué es RAG?"),
        llamar_modelo_async("¿Qué es un embedding?")
    )
    return r1, r2

print("=== Versión asíncrona ===")
inicio = time.time()
# En Jupyter se puede usar await directamente
r1, r2 = await procesar_dos_queries()
print(f"Tiempo total: {time.time() - inicio:.1f}s (procesó ambas en paralelo)")

### 3.1 Streaming — así se envían las respuestas del modelo

En el proyecto, las respuestas no se envían todas de una vez: se van enviando fragmento a fragmento  
(token a token). Esto es lo que produce el efecto de "escritura en tiempo real".

Se implementa con un **async generator**: una función async que usa `yield` en lugar de `return`.

In [ ]:
import asyncio

async def simular_streaming(texto: str, delay: float = 0.1):
    """Simula el streaming token a token de un modelo de lenguaje."""
    palabras = texto.split()
    for palabra in palabras:
        await asyncio.sleep(delay)  # simula el tiempo de generación
        yield palabra + " "        # yield envía cada fragmento

async def consumir_stream():
    respuesta_completa = ""
    print("Respuesta del modelo: ", end="", flush=True)

    # async for itera sobre un async generator
    async for fragmento in simular_streaming("RAG combina recuperación de información con generación de texto."):
        print(fragmento, end="", flush=True)  # imprime sin salto de línea
        respuesta_completa += fragmento

    print()  # salto de línea al final
    print(f"\nTexto completo recibido: {len(respuesta_completa)} caracteres")

await consumir_stream()

### → Tu turno 3.1

Escribí una corutina `procesar_documentos_async` que:
- Reciba una lista de nombres de documentos (`list[str]`)
- Para cada documento, simule una carga asíncrona con `asyncio.sleep(0.5)`
- Imprima un mensaje cuando cada documento termine de cargar
- Procese todos los documentos concurrentemente usando `asyncio.gather`
- Devuelva la cantidad de documentos procesados

In [ ]:
import asyncio

# → Tu turno: implementá la función

async def cargar_un_documento(nombre: str) -> str:
    """Carga un solo documento de forma asíncrona."""
    pass  # reemplazá este pass

async def procesar_documentos_async(documentos: list[str]) -> int:
    """Procesa todos los documentos concurrentemente."""
    pass  # reemplazá este pass


# Probá tu función:
# import time
# inicio = time.time()
# total = await procesar_documentos_async(["legal.pdf", "tecnico.txt", "faq.pdf"])
# print(f"\n{total} documentos cargados en {time.time() - inicio:.1f}s")

---
## Sección 4 — Manejo de Errores

La integración con vLLM, Chroma y modelos de embeddings falla.  
El código robusto no asume que todo funciona: anticipa y registra los fallos.

In [ ]:
import requests
from typing import Optional

def verificar_servidor(url: str, timeout: int = 5) -> tuple[bool, Optional[str]]:
    """
    Verifica si un servidor está disponible.
    
    Devuelve (True, None) si está OK.
    Devuelve (False, mensaje_de_error) si falló.
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # lanza HTTPError si el código no es 2xx
        return True, None

    except requests.exceptions.ConnectionError:
        # El servidor no está corriendo o la URL es incorrecta
        return False, f"No se pudo conectar a {url}. ¿Está el servidor levantado?"

    except requests.exceptions.Timeout:
        # El servidor tardó más de 'timeout' segundos
        return False, f"El servidor tardó más de {timeout}s en responder."

    except requests.exceptions.HTTPError as e:
        # El servidor respondió pero con un error HTTP (404, 500, etc.)
        return False, f"Error HTTP {e.response.status_code}: {e}"


# Probar con una URL que no existe
ok, error = verificar_servidor("http://localhost:9999/v1/models")
if ok:
    print("Servidor disponible.")
else:
    print(f"Servidor no disponible: {error}")

# Probar con una URL real (debería responder)
ok, error = verificar_servidor("https://httpbin.org/get")
print(f"httpbin.org: {'OK' if ok else error}")

In [ ]:
# El bloque finally: se ejecuta siempre, haya error o no
# Útil para cerrar conexiones, liberar recursos, etc.

def cargar_documento(ruta: str) -> str:
    """
    Carga un documento de texto.
    Demuestra el uso de finally para limpieza garantizada.
    """
    archivo = None
    try:
        archivo = open(ruta, 'r', encoding='utf-8')
        contenido = archivo.read()
        print(f"Documento cargado: {len(contenido)} caracteres.")
        return contenido

    except FileNotFoundError:
        print(f"Error: el archivo '{ruta}' no existe.")
        return ""

    except PermissionError:
        print(f"Error: sin permisos para leer '{ruta}'.")
        return ""

    except UnicodeDecodeError:
        print(f"Error: '{ruta}' no es un archivo de texto UTF-8 válido.")
        return ""

    finally:
        # Esto se ejecuta SIEMPRE, incluso si hubo error
        if archivo is not None:
            archivo.close()
            print("Archivo cerrado.")


# Probar con archivo inexistente
contenido = cargar_documento("archivo_que_no_existe.txt")
print(f"Contenido obtenido: '{contenido}'")

### → Tu turno 4.1

Escribí una función `dividir_texto` que:
- Reciba `texto` (str) y `chunk_size` (int)
- Divida el texto en fragmentos de `chunk_size` caracteres
- Lance un `ValueError` si `chunk_size` es menor o igual a 0 (con un mensaje descriptivo)
- Lance un `ValueError` si `texto` está vacío
- Devuelva una `list[str]` con los fragmentos

Después probá la función con entradas válidas e inválidas, capturando el `ValueError`.

In [ ]:
# → Tu turno: implementá la función

def dividir_texto(texto: str, chunk_size: int) -> list[str]:
    pass  # reemplazá este pass


# Probá con estas llamadas:
# texto_largo = "Python es un lenguaje de programación poderoso y versátil. " * 5
#
# try:
#     fragmentos = dividir_texto(texto_largo, 50)
#     print(f"Dividido en {len(fragmentos)} fragmentos.")
#     print(f"Primer fragmento: '{fragmentos[0]}'")
# except ValueError as e:
#     print(f"Error de validación: {e}")
#
# try:
#     dividir_texto(texto_largo, -10)  # debe lanzar ValueError
# except ValueError as e:
#     print(f"Error capturado correctamente: {e}")

---
## Sección 5 — Entornos Virtuales

Esta sección no tiene celdas ejecutables — se trabaja en la terminal.  
Seguí los pasos en tu terminal antes de continuar.

### Pasos obligatorios:

```bash
# 1. Crear el entorno virtual (una sola vez por proyecto)
python -m venv .venv

# 2. Activarlo (cada vez que abrís una terminal nueva)
source .venv/bin/activate       # Linux / macOS
.venv\Scripts\activate          # Windows

# 3. Verificar que está activo (debe aparecer (.venv) al inicio del prompt)
which python                    # debe apuntar a .venv/bin/python

# 4. Instalar dependencias del proyecto
pip install -r requirements.txt

# 5. Ver qué está instalado
pip list
```

### Verificación desde Python:

In [ ]:
import sys
import os

print(f"Python ejecutable: {sys.executable}")
print(f"Versión: {sys.version}")

# Verificar si estamos dentro de un entorno virtual
en_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)

if en_venv:
    print("✅ Estás dentro de un entorno virtual.")
    print(f"   Entorno: {sys.prefix}")
else:
    print("⚠️  No estás usando un entorno virtual.")
    print("   Creá uno con: python -m venv .venv")

---
## Checklist de cierre — Etapa 1

Antes de dar la etapa por completada, verificá que podés hacer esto sin ayuda:

- [ ] Escribir una clase con `__init__`, atributos y métodos propios.
- [ ] Crear una subclase que herede de una clase base con `super().__init__()`.
- [ ] Definir funciones con type hints en todos los parámetros y en el retorno.
- [ ] Escribir una corutina `async def` y llamarla con `await`.
- [ ] Usar `asyncio.gather` para ejecutar tareas concurrentemente.
- [ ] Escribir un `try/except` con al menos dos tipos de error distintos.
- [ ] Crear un entorno virtual, activarlo e instalar dependencias.
- [ ] Explicar qué es `RAGState` y por qué usa `TypedDict`.

---

> Cuando completés todos los ítems, podran pasar a la **Etapa 2: Fundamentos LLM**.